# H&M Dataset Preprocessing

**Pipeline:**
1. Load raw files
2. Domain filter (product type, purchase count, location)
3. Activity floor — drop users with < `MIN_INTERACTIONS` total purchases
4. Random 75/25 train/test split
5. Cold-start filter — remove test rows whose user is absent from train
6. Integer encoding + 80/20 validation split
7. Save to disk / reload from cache


## 0. Imports


In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)


## 1. Load Raw Files


In [5]:
DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code"

df          = pd.read_csv(os.path.join(DATA_DIR, "transactions_train.csv"),
                          dtype={"article_id": str})
customer_df = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"),
                          dtype={"article_id": str, "product_code": str})
article_df  = pd.read_csv(os.path.join(DATA_DIR, "articles.csv"),
                          dtype={"article_id": str})

df["product_code"] = df["article_id"].map(
    article_df.set_index("article_id")["product_code"])

print(f"Raw transactions : {len(df):,}")
print(f"Unique customers : {df['customer_id'].nunique():,}")


Raw transactions : 31,788,324
Unique customers : 1,362,281


## 2. Domain Filter

Restrict to the top-6 product types, customers with > 120 unique product purchases,
and customers in the top-5000 postal codes.


In [8]:
chosen_types = (
    article_df.groupby("product_type_name")["product_code"]
    .count().sort_values()[-6:].index
)
chosen_customers = (
    df.groupby("customer_id")["product_code"].nunique()
    .pipe(lambda s: s[s > 120]).index
)
location_customers = customer_df["postal_code"].value_counts()[1:5001].index
customer_df = customer_df[customer_df["postal_code"].isin(location_customers)]

chosen_articles = article_df[article_df["product_type_name"].isin(chosen_types)]
df = df[df["product_code"].isin(chosen_articles["product_code"])]
df = df[df["customer_id"].isin(chosen_customers)]
df = df[df["customer_id"].isin(customer_df["customer_id"])]

df.drop(["t_dat", "price", "article_id", "sales_channel_id"], axis=1, inplace=True)
df["bought"] = 1
df.reset_index(drop=True, inplace=True)

print(f"Filtered transactions : {len(df):,}")
print(f"Unique customers      : {df['customer_id'].nunique():,}")
print(f"Unique products       : {df['product_code'].nunique():,}")


Filtered transactions : 216,390
Unique customers      : 1,765
Unique products       : 12,782


## 3. Configuration


In [10]:
MIN_INTERACTIONS = 100     # drop users with fewer total purchases than this
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"


## 4. Run Pipeline

- **Activity floor**: keep only users with >= `MIN_INTERACTIONS` total purchases
- **75/25 split**: random row split (same ratio as ML-100K)
- **Cold-start filter**: remove any test row whose user does not appear in train
- **Encoding**: integer-encode users and items (fit on train)
- **Val split**: 80/20 of train for early stopping


In [12]:
cache_tag  = f"hm_m{MIN_INTERACTIONS}"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

if all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]):
    # ── Fast path: reload ──────────────────────────────────────────────
    print(f"Loading cached dataset: {cache_tag}")
    train_df = pd.read_csv(train_path)
    val_df   = pd.read_csv(val_path)
    test_df  = pd.read_csv(test_path)
    meta_df  = pd.read_csv(meta_path)
    N_USERS  = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
    N_ITEMS  = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])
    print(f"  Users : {N_USERS} | Items : {N_ITEMS}")
    print(f"  Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")

else:
    # ── Slow path: run pipeline ────────────────────────────────────────
    print(f"Running pipeline ({cache_tag}) ...")

    # Step 1: activity floor — keep users with >= MIN_INTERACTIONS purchases
    user_counts = (
        df.groupby("customer_id")["product_code"]
        .count()
        .rename("n_interactions")
    )
    active_users = user_counts[user_counts >= MIN_INTERACTIONS].index
    df_active    = df[df["customer_id"].isin(active_users)].copy()
    df_active.reset_index(drop=True, inplace=True)

    print(f"\nActivity floor (>= {MIN_INTERACTIONS}):")
    print(f"  Before : {df['customer_id'].nunique()} users, {len(df):,} rows")
    print(f"  After  : {df_active['customer_id'].nunique()} users, {len(df_active):,} rows")
    print(f"  Dropped: {df['customer_id'].nunique() - df_active['customer_id'].nunique()} users")

    # Step 2: random 75/25 train/test split on interaction rows
    X = df_active[["customer_id", "product_code"]].values
    y = df_active["bought"].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=SEED
    )
    train_df = pd.DataFrame(X_train, columns=["customer_id", "product_code"])
    test_df  = pd.DataFrame(X_test,  columns=["customer_id", "product_code"])
    train_df["bought"] = y_train
    test_df["bought"]  = y_test

    print(f"\n75/25 split:")
    print(f"  Train : {len(train_df):,} rows")
    print(f"  Test  : {len(test_df):,} rows")

    # Step 3: cold-start filter — test users must appear in train
    train_users = set(train_df["customer_id"])
    mask        = test_df["customer_id"].isin(train_users)
    n_dropped   = (~mask).sum()
    test_df     = test_df[mask].copy()

    print(f"\nCold-start filter:")
    print(f"  Dropped {n_dropped} test rows with users absent from train")
    print(f"  Test after filter : {len(test_df):,} rows")

    # Step 4: integer encoding (fit on combined train + test)
    all_users = np.unique(
        pd.concat([train_df[["customer_id"]], test_df[["customer_id"]]])["customer_id"]
    )
    all_items = np.unique(
        pd.concat([train_df[["product_code"]], test_df[["product_code"]]])["product_code"]
    )
    customer_id2idx = {c: i for i, c in enumerate(all_users)}
    item_id2idx     = {a: i for i, a in enumerate(all_items)}
    for df_ in [train_df, test_df]:
        df_["customer_id"]  = df_["customer_id"].map(customer_id2idx)
        df_["product_code"] = df_["product_code"].map(item_id2idx)

    N_USERS  = len(all_users)
    N_ITEMS  = len(all_items)
    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    # Step 5: 80/20 validation split of train
    n_val    = int(len(train_df) * 0.2)
    val_df   = train_df.tail(n_val).reset_index(drop=True)
    train_df = train_df.head(len(train_df) - n_val).reset_index(drop=True)

    # Save
    os.makedirs(SAMPLED_DATA_DIR, exist_ok=True)
    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path,     index=False)
    test_df.to_csv(test_path,   index=False)
    pd.DataFrame([
        {"key": "n_users",          "value": N_USERS},
        {"key": "n_items",          "value": N_ITEMS},
        {"key": "n_train",          "value": len(train_df)},
        {"key": "n_val",            "value": len(val_df)},
        {"key": "n_test",           "value": len(test_df)},
        {"key": "min_interactions", "value": MIN_INTERACTIONS},
        {"key": "seed",             "value": SEED},
    ]).to_csv(meta_path, index=False)

    density = len(train_df) / (N_USERS * N_ITEMS) * 100
    print(f"\nSaved to {SAMPLED_DATA_DIR}/")
    print(f"Users   : {N_USERS}")
    print(f"Items   : {N_ITEMS}")
    print(f"Train   : {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"Density : {density:.4f}%")


Loading cached dataset: hm_m100
  Users : 1061 | Items : 11974
  Train : 96802 | Val : 24200 | Test : 40335


## 5. Verification


In [14]:
# Every test user must appear in train
train_users = set(train_df["customer_id"].unique())
test_users  = set(test_df["customer_id"].unique())
cold_start  = test_users - train_users
print(f"Train users           : {len(train_users)}")
print(f"Test users            : {len(test_users)}")
print(f"Cold-start test users : {len(cold_start)}  (must be 0)")
assert len(cold_start) == 0, "Cold-start users found!"

# Every user must have >= MIN_INTERACTIONS in train
counts = train_df.groupby("customer_id")["product_code"].count()
below  = (counts < MIN_INTERACTIONS).sum()
print(f"Users below activity floor in train : {below}  (should be 0)")

print(f"\nInteractions per user (train):")
print(f"  Min    : {counts.min()}")
print(f"  Median : {counts.median():.0f}")
print(f"  Mean   : {counts.mean():.1f}")
print(f"  Max    : {counts.max()}")


Train users           : 1061
Test users            : 1061
Cold-start test users : 0  (must be 0)
Users below activity floor in train : 776  (should be 0)

Interactions per user (train):
  Min    : 50
  Median : 81
  Mean   : 91.2
  Max    : 325
